# 🏆 Football Score Prediction — AW-MAE Competition Pipeline

Pipeline lengkap memprediksi `team_goals` & `opp_goals` dengan metric **Augmented Weighted Mean Absolute Error**.

### Formula Metrik
$$\text{Loss}_i = \text{MAE}_i - (0.10 \cdot \text{Exact}_i + 0.05 \cdot \text{Outcome}_i + 0.05 \cdot \text{GD}_i)$$
$$\text{AW-MAE} = \frac{\sum w_i \cdot \text{Loss}_i}{\sum w_i}$$

### ✅ Hasil (OOF)
| Metric | Nilai |
|---|---|
| OOF AW-MAE | **0.95781** |
| Baseline AW-MAE | 1.35559 |
| Improvement | **+0.39778** |
| Outcome Accuracy | 57.6% |
| Exact Score Accuracy | 10.5% |

## 0. Setup & Import

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
np.random.seed(42)
print('Libraries loaded ✅')

## 1. AW-MAE Metric

In [ ]:
def get_tournament_weight(tournament: str) -> float:
    t = tournament.lower()
    major = ['world cup','copa america','copa américa','uefa euro',
             'african cup of nations','afc asian cup','gold cup','olympic','nations cup']
    for kw in major:
        if kw in t and 'qualif' not in t:
            return 1.5
    if 'qualif' in t or 'qualification' in t:
        return 1.0
    if 'friendly' in t:
        return 0.8
    return 1.2


def awmae_score(ytg, yog, ptg, pog, w, l1=0.10, l2=0.05, l3=0.05):
    ytg, yog = np.array(ytg), np.array(yog)
    ptg = np.round(np.array(ptg)).clip(0)
    pog = np.round(np.array(pog)).clip(0)
    w   = np.array(w)
    mae = (np.abs(ytg - ptg) + np.abs(yog - pog)) / 2
    exact   = ((ptg == ytg) & (pog == yog)).astype(float)
    outcome = (np.sign(ytg - yog) == np.sign(ptg - pog)).astype(float)
    gd_b    = ((ytg - yog) == (ptg - pog)).astype(float)
    loss    = mae - (l1 * exact + l2 * outcome + l3 * gd_b)
    return np.sum(w * loss) / np.sum(w)

print('AW-MAE (perfect):', awmae_score([2,1],[1,0],[2,1],[1,0],[1.5,1.0]))
print('AW-MAE (zeros):  ', awmae_score([2,1],[1,0],[0,0],[0,0],[1.5,1.0]))

## 2. Load & EDA

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sub   = pd.read_csv('sample submission.csv')

print(f'Train: {train.shape}  ({train["date"].min()} → {train["date"].max()})')
print(f'Test:  {test.shape}  ({test["date"].min()} → {test["date"].max()})')
print(f'Targets: team_goals mean={train["team_goals"].mean():.2f}, opp_goals mean={train["opp_goals"].mean():.2f}')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ['team_goals', 'opp_goals']):
    counts = train[col].value_counts().sort_index()
    ax.bar(counts.index[:13], counts.values[:13], color='#4C72B0', edgecolor='white')
    ax.axvline(train[col].mean(), color='crimson', linestyle='--', label=f'Mean={train[col].mean():.2f}')
    ax.set_title(f'{col} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Goals'); ax.set_ylabel('Count'); ax.legend()
plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
train['tournament_weight'] = train['tournament'].apply(get_tournament_weight)
test['tournament_weight']  = test['tournament'].apply(get_tournament_weight)

print('Avg goals per tournament weight:')
print(train.groupby('tournament_weight')[['team_goals','opp_goals']].mean().round(3))
print('\nWeight distribution:',dict(train['tournament_weight'].value_counts()))

## 3. Feature Engineering — Rolling Stats

> **Challenge:** Test set tidak memiliki performance features (elo, rank, rolling stats karena data test adalah pertandingan 2011–2026, sedangkan train hanya sampai 2011). Features direkonstruksi dari histori train menggunakan `pd.merge_asof` (efficient time-aware lookup, tanpa data leakage).

In [ ]:
def build_history(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    rows = []
    for team_col, gs_col, gc_col in [('team','team_goals','opp_goals'),('opponent','opp_goals','team_goals')]:
        tmp = df[[team_col,'date',gs_col,gc_col]].copy()
        tmp.columns = ['team','date','gs','gc']
        tmp['gd']  = tmp['gs'] - tmp['gc']
        tmp['pts'] = np.where(tmp['gs']>tmp['gc'],3,np.where(tmp['gs']==tmp['gc'],1,0))
        tmp['win'] = (tmp['pts']==3).astype(int)
        rows.append(tmp)
    return pd.concat(rows).sort_values('date').reset_index(drop=True)

def compute_rolling(hist, n):
    out = hist.sort_values(['team','date']).copy()
    g = out.groupby('team')
    out[f'pts_{n}'] = g['pts'].transform(lambda x: x.shift(1).rolling(n, min_periods=1).sum())
    out[f'gd_{n}']  = g['gd'].transform(lambda x: x.shift(1).rolling(n, min_periods=1).sum())
    out[f'gs_{n}']  = g['gs'].transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    out[f'gc_{n}']  = g['gc'].transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    out[f'win_{n}'] = g['win'].transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    return out.sort_values('date')[['date','team',f'pts_{n}',f'gd_{n}',f'gs_{n}',f'gc_{n}',f'win_{n}']]

def merge_rolling(target_df, r5, r10, team_col, prefix):
    orig_idx = target_df.index
    tdf = target_df[['date',team_col]].rename(columns={team_col:'team'}).sort_values('date')
    m5  = pd.merge_asof(tdf, r5.sort_values('date'),  on='date', by='team', direction='backward')
    m10 = pd.merge_asof(tdf, r10.sort_values('date'), on='date', by='team', direction='backward')
    res = pd.DataFrame(index=tdf.index)
    res[f'{prefix}_points_last5']       = m5['pts_5'].values
    res[f'{prefix}_gd_last5']           = m5['gd_5'].values
    res[f'{prefix}_avg_goals_last5']    = m5['gs_5'].values
    res[f'{prefix}_avg_conceded_last5'] = m5['gc_5'].values
    res[f'{prefix}_win_rate_last10']    = m10['win_10'].values
    res[f'{prefix}_points_last10']      = m10['pts_10'].values
    res.index = orig_idx[tdf.index]
    return res.sort_index()

print('Feature engineering functions defined ✅')

In [ ]:
%%time
print('Building match history from train...')
hist = build_history(train)
r5  = compute_rolling(hist, 5)
r10 = compute_rolling(hist, 10)
print(f'History: {len(hist):,} rows  |  r5: {r5.shape}  |  r10: {r10.shape}')

In [ ]:
%%time
test_c  = test.copy();  test_c['date']  = pd.to_datetime(test_c['date']);  test_c  = test_c.sort_values('date')
train_c = train.copy(); train_c['date'] = pd.to_datetime(train_c['date']); train_c = train_c.sort_values('date')

te_roll = pd.concat([merge_rolling(test_c, r5, r10, 'team','team'), merge_rolling(test_c, r5, r10, 'opponent','opp')], axis=1)
tr_roll = pd.concat([merge_rolling(train_c, r5, r10, 'team','team'), merge_rolling(train_c, r5, r10, 'opponent','opp')], axis=1)
for df in [te_roll, tr_roll]:
    df['points_last5_diff'] = df['team_points_last5'] - df['opp_points_last5']
    df['gd_last5_diff']     = df['team_gd_last5'] - df['opp_gd_last5']

print(f'te_roll: {te_roll.shape},  tr_roll: {tr_roll.shape}')

In [ ]:
ORIG = ['team_points_last5','team_gd_last5','team_avg_goals_last5','team_avg_conceded_last5',
        'team_win_rate_last10','team_points_last10','opp_points_last5','opp_gd_last5',
        'opp_avg_goals_last5','opp_avg_conceded_last5','opp_win_rate_last10','opp_points_last10',
        'points_last5_diff','gd_last5_diff']
train_m = train.drop(columns=[c for c in ORIG if c in train.columns]).join(tr_roll)
test_m  = test.join(te_roll)
for col in ['elo_team','elo_opponent','rank_team','rank_opponent','rank_diff',
            'h2h_points_last5','h2h_gd_last5']:
    test_m[col] = np.nan
test_m['rank_missing_team'] = 1
test_m['rank_missing_opp']  = 1
print(f'train_m: {train_m.shape},  test_m: {test_m.shape}')

## 4. Preprocessing

In [ ]:
def add_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['is_recent'] = (df['year'] >= 2000).astype(int)
    cmap = {'UEFA':0,'CAF':1,'CONMEBOL':2,'AFC':3,'CONCACAF':4,'OFC':5}
    df['conf_team_enc'] = df['confederation_team'].map(cmap).fillna(6)
    df['conf_opp_enc']  = df['confederation_opp'].map(cmap).fillna(6)
    df['same_conf']     = (df['confederation_team'] == df['confederation_opp']).astype(int)
    df['gender_enc']    = (df['gender'] == 'M').astype(int)
    df['altitude_venue'] = df['altitude_venue'].clip(lower=0).fillna(0)
    for col in ['population_team','population_opp','gdp_per_capita_team','gdp_per_capita_opp']:
        df[col] = df[col].fillna(df[col].median())
    df['log_pop_team']  = np.log1p(df['population_team'])
    df['log_pop_opp']   = np.log1p(df['population_opp'])
    df['log_gdp_team']  = np.log1p(df['gdp_per_capita_team'])
    df['log_gdp_opp']   = np.log1p(df['gdp_per_capita_opp'])
    df['gdp_ratio']     = df['gdp_per_capita_team'] / (df['gdp_per_capita_opp'] + 1)
    df['travel_diff']   = df['distance_travel_team'].fillna(0) - df['distance_travel_opp'].fillna(0)
    df['temperature_venue'] = df['temperature_venue'].fillna(df['temperature_venue'].median())
    return df

train_p = add_features(train_m)
test_p  = add_features(test_m)

all_teams = pd.concat([train_p['team'], train_p['opponent']]).unique()
le = LabelEncoder().fit(all_teams)
known = set(le.classes_)
train_p['team_enc'] = le.transform(train_p['team'])
train_p['opponent_enc'] = le.transform(train_p['opponent'])
test_p['team_enc']  = test_p['team'].apply(lambda x: le.transform([x])[0] if x in known else -1)
test_p['opponent_enc'] = test_p['opponent'].apply(lambda x: le.transform([x])[0] if x in known else -1)
print('Preprocessing done ✅')

## 5. Feature Selection

In [ ]:
FEATURE_COLS = [
    'is_home','neutral','tournament_weight','year','month','is_recent','gender_enc',
    'team_enc','opponent_enc','conf_team_enc','conf_opp_enc','same_conf',
    'team_points_last5','team_gd_last5','team_avg_goals_last5','team_avg_conceded_last5',
    'team_win_rate_last10','team_points_last10',
    'opp_points_last5','opp_gd_last5','opp_avg_goals_last5','opp_avg_conceded_last5',
    'opp_win_rate_last10','opp_points_last10',
    'points_last5_diff','gd_last5_diff',
    'h2h_points_last5','h2h_gd_last5',
    'elo_team','elo_opponent','rank_team','rank_opponent','rank_diff',
    'rank_missing_team','rank_missing_opp',
    'log_pop_team','log_pop_opp','log_gdp_team','log_gdp_opp',
    'gdp_ratio','travel_diff','altitude_venue','temperature_venue',
]
for col in FEATURE_COLS:
    if col not in train_p.columns: train_p[col] = np.nan
    if col not in test_p.columns:  test_p[col]  = np.nan

X_train = train_p[FEATURE_COLS]
X_test  = test_p[FEATURE_COLS]
y_team  = train_p['team_goals'].values
y_opp   = train_p['opp_goals'].values
weights = train_p['tournament_weight'].values
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}, Features: {len(FEATURE_COLS)}')

## 6. Model Training — LightGBM 5-Fold CV

In [ ]:
LGB_PARAMS = dict(
    objective='regression', metric='mae', learning_rate=0.05,
    num_leaves=127, min_child_samples=50, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=5, reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=2000, random_state=42, n_jobs=-1, verbose=-1
)

def train_lgb_cv(X, y, w, Xte, label, n_splits=5):
    kf  = KFold(n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(X))
    tp  = np.zeros(len(Xte))
    last_model = None
    for fold, (tri, vi) in enumerate(kf.split(X)):
        m = lgb.LGBMRegressor(**LGB_PARAMS)
        m.fit(X.iloc[tri], y[tri], sample_weight=w[tri],
              eval_set=[(X.iloc[vi], y[vi])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
        oof[vi] = m.predict(X.iloc[vi])
        tp += m.predict(Xte) / n_splits
        last_model = m
        print(f'  [{label}] Fold {fold+1}/{n_splits}  MAE={mean_absolute_error(y[vi], oof[vi]):.4f}  best_iter={m.best_iteration_}')
    print(f'  → OOF MAE: {mean_absolute_error(y, oof):.4f}\n')
    return oof, tp, last_model

print('Ready ✅')

In [ ]:
%%time
print('TRAINING: team_goals')
oof_team, pred_team, model_team = train_lgb_cv(X_train, y_team, weights, X_test, 'team_goals')

In [ ]:
%%time
print('TRAINING: opp_goals')
oof_opp, pred_opp, model_opp = train_lgb_cv(X_train, y_opp, weights, X_test, 'opp_goals')

## 7. Evaluasi OOF (AW-MAE)

In [ ]:
score    = awmae_score(y_team, y_opp, oof_team, oof_opp, weights)
base_tg  = np.full(len(y_team), round(y_team.mean()))
base_og  = np.full(len(y_opp), round(y_opp.mean()))
baseline = awmae_score(y_team, y_opp, base_tg, base_og, weights)

tg_r = np.round(oof_team).clip(0)
og_r = np.round(oof_opp).clip(0)
outcome_acc = (np.sign(y_team-y_opp)==np.sign(tg_r-og_r)).mean()
exact_acc   = ((tg_r==y_team)&(og_r==y_opp)).mean()
gd_acc      = ((tg_r-og_r)==(y_team-y_opp)).mean()

print('┌──────────────────────────────────────────┐')
print(f'│  OOF AW-MAE:            {score:.5f}         │')
print(f'│  Baseline AW-MAE:       {baseline:.5f}         │')
print(f'│  Improvement:           {baseline-score:+.5f}         │')
print('├──────────────────────────────────────────┤')
print(f'│  Outcome Accuracy:      {outcome_acc:.4f}           │')
print(f'│  Exact Score Accuracy:  {exact_acc:.4f}           │')
print(f'│  Goal Diff Accuracy:    {gd_acc:.4f}           │')
print('└──────────────────────────────────────────┘')

In [ ]:
print('AW-MAE per tournament weight:')
for wval, wname in [(0.8,'Friendly'),(1.0,'Qualifier'),(1.2,'Regional'),(1.5,'Major')]:
    mask = weights == wval
    if mask.sum()==0: continue
    s = awmae_score(y_team[mask],y_opp[mask],oof_team[mask],oof_opp[mask],np.ones(mask.sum()))
    print(f'  {wname:12s} (w={wval}) | N={mask.sum():6d} | AW-MAE={s:.4f}')

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, model, title in zip(axes, [model_team, model_opp], ['team_goals', 'opp_goals']):
    fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': model.feature_importances_})
    fi = fi.sort_values('importance').tail(20)
    ax.barh(fi['feature'], fi['importance'], color='#4C72B0')
    ax.set_title(f'Top 20 Features — {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, true, pred, lbl in zip(axes, [y_team, y_opp], [tg_r, og_r], ['team_goals', 'opp_goals']):
    errs = np.abs(pred - true)
    pd.Series(errs).value_counts().sort_index()[:8].plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(f'{lbl} — Absolute Error (OOF)', fontweight='bold')
    ax.set_xlabel('|Error|'); ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 8. Generate Submission

In [ ]:
final_team = np.round(pred_team).astype(int).clip(0)
final_opp  = np.round(pred_opp).astype(int).clip(0)

submission = pd.DataFrame({'Id': test['Id'], 'team_goals': final_team, 'opp_goals': final_opp})
assert list(submission.columns) == list(sub.columns)
assert len(submission) == len(sub)
assert submission[['team_goals','opp_goals']].min().min() >= 0

submission.to_csv('submission.csv', index=False)
print(f'✅ submission.csv saved! Shape: {submission.shape}')
submission.head(10)

## 9. Summary

In [ ]:
print('='*60)
print('  PIPELINE SUMMARY')
print('='*60)
print(f'  Train rows:           {len(train):>10,}')
print(f'  Test rows:            {len(test):>10,}')
print(f'  Features:             {len(FEATURE_COLS):>10}')
print(f'  Model:                LightGBM (5-Fold CV)')
print(f'  Sample weight:        tournament_weight')
print('-'*60)
print(f'  OOF AW-MAE:           {score:>10.5f}')
print(f'  Baseline AW-MAE:      {baseline:>10.5f}')
print(f'  Improvement:          {baseline-score:>+10.5f}')
print(f'  Outcome Accuracy:     {outcome_acc:>10.4f}')
print(f'  Exact Score Accuracy: {exact_acc:>10.4f}')
print('='*60)

---
## 📌 Ide Improvement
- **Poisson objective** — goal scoring mengikuti distribusi Poisson
- **Ensemble** LightGBM + XGBoost + CatBoost
- **H2H features** lengkap dari seluruh histori pertemuan
- **Elo rekonstruksi** untuk test set berdasarkan hasil match secara sekuensial
- **Post-processing** berbasis distribusi Poisson untuk score rounding yang lebih optimal